# Filter AmericanStories articles (spec: `metaphor/sample_article.md`)

Filter the newspaper corpus to articles **published 1900-01-01 .. 1960-12-31** that match (OR logic)
any of the keywords **federal / administration / government / new deal / the feds**, preserving all
metadata and the full article text (no paraphrasing).

**Decisions (per spec + confirmed):**
- *Matching = stem / left-boundary* (spec default): `\bfederal` also matches *federalism*, `\bgovernment`
  matches *governmental*. For exact whole-word instead, change a pattern to e.g. `\bfederal\b`.
- Matched against **article body and headline**, case-insensitive (RE2 `\b`, `\s+`, IGNORECASE).
- `new deal` = `\bnew\s+deal` (1+ whitespace); `the feds` = `\bthe\s+feds`.

**Field mapping:** `text`->`article`; `date` is standard `YYYY-MM-DD`; `headline`,`byline`,`page`,`edition`,
`article_id` are native. `region/state/city`,`word_count`,`url` are NOT native -> `state` is best-effort
joined from `newspaper_metainfo.csv`, `word_count` is derived; unmatched `state` = null.

**Outputs:** `metaphor/output/filtered_articles.parquet` + `metaphor/output/run_summary.json`.
Processing streams **one year-split at a time** (memory-mapped Arrow; rows written incrementally via
`ParquetWriter`) so the full corpus is never held in memory.

In [1]:
import os, re, json, time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from datasets import load_dataset

CACHE_DIR = "/kellogg/proj/dashun/ruipan/fame/data/raw_historical_newspaper"
META_CSV  = "/kellogg/proj/dashun/ruipan/fame/data/newspaper_meta/newspaper_metainfo.csv"
OUT_DIR   = "/kellogg/proj/dashun/ruipan/fame/metaphor/output"
os.makedirs(OUT_DIR, exist_ok=True)
DATA_DIR    = "/kellogg/proj/dashun/ruipan/fame/metaphor/data"
os.makedirs(DATA_DIR, exist_ok=True)
PARQUET_OUT = os.path.join(DATA_DIR, "filtered_articles.parquet")
SUMMARY_OUT = os.path.join(OUT_DIR, "run_summary.json")

DATE_MIN = pd.Timestamp("1900-01-01")
DATE_MAX = pd.Timestamp("1960-12-31")
YEARS    = range(1900, 1961)          # year-splits to scan (date filter applied per-row too)

# keyword -> RE2 pattern. Stem/left-boundary matching (spec default).
KEYWORD_PATTERNS = {
    "federal":        r"\bfederal",
    "administration": r"\badministration",
    "government":     r"\bgovernment",
    "new deal":       r"\bnew\s+deal",
    "the feds":       r"\bthe\s+feds",
}
KEYS = list(KEYWORD_PATTERNS)
print("output dir:", OUT_DIR)

output dir: /kellogg/proj/dashun/ruipan/fame/metaphor/output


## 1. Inspect the dataset schema (before filtering)

In [2]:
dataset = load_dataset("dell-research-harvard/AmericanStories", "all_years",
                       cache_dir=CACHE_DIR, revision="e26b7f7f48c6")
splits = sorted(int(y) for y in dataset.keys())
feat = dataset[str(splits[0])].features

print("SCHEMA")
print(f"  year-splits : {len(splits)}  ({min(splits)}-{max(splits)})")
print("  fields (one row = one article):")
for n, f in feat.items():
    print(f"    - {n:15s} {f.dtype}")
print(f"  total rows (all years)          : {sum(dataset[str(y)].num_rows for y in splits):,}")
print(f"  rows in scan range {min(YEARS)}-{max(YEARS)} : {sum(dataset[str(y)].num_rows for y in YEARS):,}")

print("\nSAMPLE ROWS (1905):")
display(pd.DataFrame(dataset['1905'][:3])[['article_id','newspaper_name','date','page','headline','byline','article']]
        .assign(article=lambda d: d['article'].str.slice(0, 90) + '...'))

SCHEMA
  year-splits : 167  (1774-1963)
  fields (one row = one article):
    - article_id      string
    - newspaper_name  string
    - edition         string
    - date            string
    - page            string
    - headline        string
    - byline          string
    - article         string
  total rows (all years)          : 77,887,521
  rows in scan range 1900-1960 : 58,693,403

SAMPLE ROWS (1905):


,article_id,newspaper_name,date,page,headline,byline,article
0,1_1905-10-28_p5_sn96060765_00383341851_1905102...,The Winslow mail.,1905-10-28,p5,,,are enjoying great progress.\nAnd it is for ju...
1,2_1905-10-28_p5_sn96060765_00383341851_1905102...,The Winslow mail.,1905-10-28,p5,Congressman Il wney Talks,,A dispatch from Kansas City\nOctober Dist says...
2,3_1905-10-28_p5_sn96060765_00383341851_1905102...,The Winslow mail.,1905-10-28,p5,,,son. A full and entirely practi-\ncal saw mill...


## 2. Newspaper -> state lookup (best-effort enrichment)

In [3]:
_meta = pd.read_csv(META_CSV)
title_to_state = dict(zip(_meta["Title_processed"], _meta["State"]))
print(f"loaded {len(title_to_state):,} title->state entries from newspaper_metainfo.csv")

loaded 3,680 title->state entries from newspaper_metainfo.csv


## 3. Per-year processing

For each year split: parse dates & apply the date filter (cheap) -> vectorized keyword match on body+headline
(boolean for `matched_keywords`, occurrence counts for `match_count`) -> keep rows matching the date range AND
>= 1 keyword -> attach derived columns -> append to the Parquet file.

In [4]:
OUTPUT_SCHEMA = pa.schema([
    ("article_id", pa.string()), ("newspaper_name", pa.string()), ("edition", pa.string()),
    ("date", pa.string()), ("year", pa.int32()), ("page", pa.string()),
    ("headline", pa.string()), ("byline", pa.string()), ("article", pa.string()),
    ("word_count", pa.int32()), ("state", pa.string()),
    ("matched_keywords", pa.list_(pa.string())), ("match_count", pa.int32()),
])
OUT_COLS = [f.name for f in OUTPUT_SCHEMA]

def process_year(year):
    """Return (kept_dataframe, stats) for one year split."""
    t = dataset[str(year)].data.table          # memory-mapped pyarrow.Table
    art, hl = t.column("article"), t.column("headline")
    n = t.num_rows

    # --- date filter (cheap) ---
    s  = t.column("date").to_pandas()
    dt = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
    yr = pd.to_numeric(s.str.extract(r"(\d{4})", expand=False), errors="coerce")  # year-only fallback
    in_range = ((dt.notna() & (dt >= DATE_MIN) & (dt <= DATE_MAX)) |
                (dt.isna()  & yr.between(1900, 1960))).to_numpy()
    year_vals = dt.dt.year.where(dt.notna(), yr).to_numpy()

    # --- keyword filter (expensive), vectorized over body + headline ---
    bool_by_kw, total_count = {}, np.zeros(n, dtype=np.int64)
    for k, p in KEYWORD_PATTERNS.items():
        b = pc.or_(pc.match_substring_regex(art, p, ignore_case=True),
                   pc.match_substring_regex(hl,  p, ignore_case=True)).to_numpy(zero_copy_only=False)
        c = pc.add(pc.count_substring_regex(art, p, ignore_case=True),
                   pc.count_substring_regex(hl,  p, ignore_case=True)).to_numpy(zero_copy_only=False)
        bool_by_kw[k] = b
        total_count  += c.astype(np.int64)
    any_match = np.zeros(n, dtype=bool)
    for k in KEYS:
        any_match |= bool_by_kw[k]

    keep = in_range & any_match
    ki = np.where(keep)[0]

    sub = t.take(pa.array(ki)).to_pandas()
    sub["year"]        = year_vals[ki].astype("int32")
    sub["word_count"]  = sub["article"].str.split().str.len().fillna(0).astype("int32")
    sub["state"]       = sub["newspaper_name"].map(title_to_state)
    bm = np.vstack([bool_by_kw[k][ki] for k in KEYS]).T if len(ki) else np.empty((0, len(KEYS)), bool)
    sub["matched_keywords"] = [[KEYS[j] for j in np.where(row)[0]] for row in bm]
    sub["match_count"] = total_count[ki].astype("int32")

    obs = dt.to_numpy()[keep]
    obs = obs[~pd.isna(obs)]
    stats = dict(
        scanned=int(n),
        dropped_date=int((~in_range).sum()),
        dropped_no_keyword=int((in_range & ~any_match).sum()),
        kept=int(keep.sum()),
        kw_hits={k: int(bool_by_kw[k][ki].sum()) for k in KEYS},
        obs_min=(pd.Timestamp(obs.min()).date().isoformat() if obs.size else None),
        obs_max=(pd.Timestamp(obs.max()).date().isoformat() if obs.size else None),
    )
    return sub[OUT_COLS], stats

## 4. Run the filter over all years

In [5]:
agg = dict(total_rows_scanned=0, rows_kept=0, rows_dropped_date=0, rows_dropped_no_keyword=0,
           per_keyword_hits={k: 0 for k in KEYS})
obs_dates = []
writer = None
t0 = time.time()

for y in YEARS:
    ty = time.time()
    df, st = process_year(y)
    if len(df):
        tbl = pa.Table.from_pandas(df, schema=OUTPUT_SCHEMA, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(PARQUET_OUT, OUTPUT_SCHEMA, compression="snappy")
        writer.write_table(tbl)
    agg["total_rows_scanned"]      += st["scanned"]
    agg["rows_kept"]               += st["kept"]
    agg["rows_dropped_date"]       += st["dropped_date"]
    agg["rows_dropped_no_keyword"] += st["dropped_no_keyword"]
    for k in KEYS:
        agg["per_keyword_hits"][k] += st["kw_hits"][k]
    if st["obs_min"]: obs_dates += [st["obs_min"], st["obs_max"]]
    print(f"  {y}: scanned {st['scanned']:>9,}  kept {st['kept']:>7,}  "
          f"(drop_date {st['dropped_date']:,}, drop_nokw {st['dropped_no_keyword']:,})  [{time.time()-ty:4.1f}s]")

if writer is not None:
    writer.close()
elapsed = time.time() - t0
print(f"\nDONE in {elapsed/60:.1f} min  ->  {PARQUET_OUT}")

  1900: scanned 1,118,970  kept  56,565  (drop_date 0, drop_nokw 1,062,405)  [29.5s]


  1901: scanned 1,468,648  kept  67,502  (drop_date 0, drop_nokw 1,401,146)  [40.3s]


  1902: scanned 1,417,935  kept  61,434  (drop_date 0, drop_nokw 1,356,501)  [37.1s]


  1903: scanned 1,319,686  kept  52,667  (drop_date 0, drop_nokw 1,267,019)  [33.6s]


  1904: scanned 1,340,868  kept  50,992  (drop_date 0, drop_nokw 1,289,876)  [33.1s]


...

  1956: scanned    98,035  kept   5,742  (drop_date 0, drop_nokw 92,293)  [ 2.4s]


  1957: scanned    93,543  kept   6,185  (drop_date 0, drop_nokw 87,358)  [ 2.5s]


  1958: scanned    98,688  kept   6,255  (drop_date 0, drop_nokw 92,433)  [ 2.4s]


  1959: scanned   129,452  kept   8,869  (drop_date 0, drop_nokw 120,583)  [ 3.1s]


  1960: scanned   344,550  kept  26,331  (drop_date 0, drop_nokw 318,219)  [ 8.3s]

DONE in 25.5 min  ->  /kellogg/proj/dashun/ruipan/fame/metaphor/output/filtered_articles.parquet


## 5. Write `run_summary.json` and report

In [6]:
summary = {
    "total_rows_scanned":      agg["total_rows_scanned"],
    "rows_kept":               agg["rows_kept"],
    "rows_dropped_date":       agg["rows_dropped_date"],
    "rows_dropped_no_keyword": agg["rows_dropped_no_keyword"],
    "per_keyword_hits":        agg["per_keyword_hits"],   # # of kept articles matching each keyword
    "observed_date_range":     {"min": min(obs_dates) if obs_dates else None,
                                 "max": max(obs_dates) if obs_dates else None},
    "match_mode":              "stem/left-boundary (case-insensitive); body+headline",
    "keyword_patterns":        KEYWORD_PATTERNS,
    "scan_year_range":         [min(YEARS), max(YEARS)],
    "elapsed_seconds":         round(elapsed, 1),
    "output_parquet":          PARQUET_OUT,
}
with open(SUMMARY_OUT, "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

{
  "total_rows_scanned": 58693403,
  "rows_kept": 3615699,
  "rows_dropped_date": 0,
  "rows_dropped_no_keyword": 55077704,
  "per_keyword_hits": {
    "federal": 1051742,
    "administration": 638906,
    "government": 2430473,
    "new deal": 55288,
    "the feds": 4365
  },
  "observed_date_range": {
    "min": "1900-01-01",
    "max": "1960-12-31"
  },
  "match_mode": "stem/left-boundary (case-insensitive); body+headline",
  "keyword_patterns": {
    "federal": "\\bfederal",
    "administration": "\\badministration",
    "government": "\\bgovernment",
    "new deal": "\\bnew\\s+deal",
    "the feds": "\\bthe\\s+feds"
  },
  "scan_year_range": [
    1900,
    1960
  ],
  "elapsed_seconds": 1529.5,
  "output_parquet": "/kellogg/proj/dashun/ruipan/fame/metaphor/output/filtered_articles.parquet"
}


## 6. Verify output

In [7]:
res = pd.read_parquet(PARQUET_OUT)
print(f"filtered_articles.parquet: {len(res):,} rows x {res.shape[1]} cols")
print("columns:", list(res.columns))
print("\nmatched_keywords value counts (top combos):")
print(res["matched_keywords"].map(tuple).value_counts().head(10).to_string())
print("\nsample kept rows:")
display(res[["date","year","newspaper_name","state","headline","matched_keywords","match_count","word_count"]].head(8))

filtered_articles.parquet: 3,615,699 rows x 13 cols
columns: ['article_id', 'newspaper_name', 'edition', 'date', 'year', 'page', 'headline', 'byline', 'article', 'word_count', 'state', 'matched_keywords', 'match_count']

matched_keywords value counts (top combos):


matched_keywords
(government,)                            1970999
(federal,)                                682600
(administration,)                         418239
(federal, government)                     277841
(administration, government)              126758
(federal, administration)                  40976
(federal, administration, government)      38642
(new deal,)                                28243
(government, new deal)                      7067
(administration, new deal)                  6729

sample kept rows:


,date,year,newspaper_name,state,headline,matched_keywords,match_count,word_count
0,1900-03-23,1900,Evening star.,District of Columbia,,[government],1,497
1,1900-03-23,1900,Evening star.,District of Columbia,,[government],1,471
2,1900-03-23,1900,Evening star.,District of Columbia,,"[federal, government]",2,364
3,1900-03-23,1900,Evening star.,District of Columbia,No Tax-Ridden Washington.,[government],1,85
4,1900-03-23,1900,Evening star.,District of Columbia,\n\n,[government],1,83
5,1900-12-13,1900,Audubon Republican.,Iowa,NEWS IN GENERAL\n\nMINISTER'S ABANDON POSTS\n\...,[government],1,220
6,1900-12-13,1900,Audubon Republican.,Iowa,GREAT BRITAIN IS FIRM.\n\nNo UItimatum Could F...,[government],6,371
7,1900-12-13,1900,Audubon Republican.,Iowa,PEACE TERMS ARE FIXED.\n\nAmerican View Prevai...,[government],2,208
